# BindCraft: active GLP-1 conformer-selective binder design

<img src="https://github.com/martinpacesa/BindCraft/blob/main/pipeline.png?raw=true">

This notebook preserves the original BindCraft de novo binder workflow (AlphaFold2 backpropagation, ProteinMPNN, PyRosetta scoring, filtering, ranking, and Google Drive outputs) while adding the active-GLP-1 design objective: N-terminal hotspot engagement, positive-conformer robustness, and explicit negative-target counter-screening against GLP-1(9-36), GIP, glucagon, and oxyntomodulin.
The notebook implements the document's primary de novo mini-protein route. The optional VHH-scaffold branch is intentionally left to a separate scaffold-design workflow, because the supplied BindCraft notebook has no fixed-VHH scaffold interface and adding one would replace its core architecture rather than make a minimal objective change.


In [ ]:
#@title Installation
%%time
import os, time, gc, io
import contextlib
import json
from datetime import datetime
from ipywidgets import HTML, VBox
from IPython.display import display

if not os.path.isfile("bindcraft/params/done.txt"):
  print("Installing required BindCraft components")

  print("Pulling BindCraft code from Github")
  os.makedirs('/content/bindcraft/', exist_ok=True)
  !git clone https://github.com/martinpacesa/BindCraft /content/bindcraft/
  os.system("chmod +x /content/bindcraft/functions/dssp")
  os.system("chmod +x /content/bindcraft/functions/DAlphaBall.gcc")

  print("Installing ColabDesign")
  os.system("(mkdir bindcraft/params; apt-get install aria2 -qq; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
  tar -xf alphafold_params_2022-12-06.tar -C bindcraft/params; touch bindcraft/params/done.txt )&")
  os.system("pip install git+https://github.com/sokrypton/ColabDesign.git")
  # for debugging purposes
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

  print("Installing PyRosetta")
  !pip install -q pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release.cxx11thread.serialization

  # download params
  if not os.path.isfile("bindcraft/params/done.txt"):
    print("downloading AlphaFold params")
    while not os.path.isfile("bindcraft/params/done.txt"):
      time.sleep(5)

  print("BindCraft installation is finished, ready to run!")
else:
  print("BindCraft components already installed, ready to run!")

In [ ]:
#@title Mount your Google Drive to save design results
from google.colab import drive
drive.mount('/content/drive')
currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Google drive mounted at: {currenttime}")

bindcraft_google_drive = '/content/drive/My Drive/BindCraft/'
os.makedirs(bindcraft_google_drive, exist_ok=True)
print("BindCraft folder successfully created in your drive!")

In [ ]:
#@title Binder design settings
# @markdown ---
# @markdown Results retain the original BindCraft directory layout. The default campaign targets active GLP-1(7-36)NH2 and saves to Google Drive.
design_path = "/content/drive/MyDrive/BindCraft/GLP1_active/" # @param {"type":"string","placeholder":"/content/drive/MyDrive/BindCraft/GLP1_active/"}
binder_name = "GLP1_active" # @param {"type":"string","placeholder":"GLP1_active"}

# @markdown Binder length range for the primary de novo mini-protein route described in the technical document.
lengths = "60,120" # @param {"type":"string","placeholder":"60,120"}

# @markdown How many binder designs passing both the original BindCraft filters and the GLP-1 selectivity gate are required?
number_of_final_designs = 100 # @param {"type":"integer","placeholder":"100"}

# @markdown Selectivity gate. These are computational screening thresholds, not substitutes for the document's experimental KD and >=20-fold enrichment criteria.
minimum_positive_iptm = 0.50 # @param {"type":"number"}
maximum_negative_iptm = 0.35 # @param {"type":"number"}
minimum_selectivity_margin = 0.15 # @param {"type":"number"}
minimum_n_terminal_contacts = 2 # @param {"type":"integer"}
n_terminal_contact_cutoff = 5.0 # @param {"type":"number"}
selectivity_validation_models = 1 # @param {"type":"integer"}

# @markdown The default panel is downloaded from RCSB PDB and reduced to peptide-only PDB files. Edit the source list below only when using a validated in-house structural ensemble.
target_cache_dir = "/content/bindcraft/glp1_target_panel"
os.makedirs(target_cache_dir, exist_ok=True)

def _download_pdb(pdb_id):
    import urllib.request
    output_path = os.path.join(target_cache_dir, f"{pdb_id.upper()}.pdb")
    if not os.path.isfile(output_path):
        url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
        print(f"Downloading {pdb_id.upper()} from RCSB PDB")
        urllib.request.urlretrieve(url, output_path)
    return output_path

def _extract_chain(source_pdb, source_chain, output_name, model_index=1, drop_first_residues=0):
    output_path = os.path.join(target_cache_dir, output_name)
    if os.path.isfile(output_path):
        return output_path

    selected_lines = []
    residue_order = []
    seen_residues = set()
    current_model = 1
    has_models = False
    with open(source_pdb, "r") as handle:
        for line in handle:
            if line.startswith("MODEL"):
                has_models = True
                try:
                    current_model = int(line.split()[1])
                except (IndexError, ValueError):
                    current_model += 1
                continue
            if line.startswith("ENDMDL"):
                continue
            if has_models and current_model != model_index:
                continue
            if not line.startswith("ATOM") or len(line) < 54 or line[21].strip() != source_chain:
                continue
            residue_key = (line[22:26], line[26])
            if residue_key not in seen_residues:
                seen_residues.add(residue_key)
                residue_order.append(residue_key)
            selected_lines.append((residue_key, line))

    kept_residues = set(residue_order[drop_first_residues:])
    kept_lines = [line for residue_key, line in selected_lines if residue_key in kept_residues]
    if not kept_lines:
        raise ValueError(f"No atoms retained for chain {source_chain} in {source_pdb}")
    with open(output_path, "w") as handle:
        handle.writelines(kept_lines)
        handle.write("TER\nEND\n")
    return output_path

def _first_residue_hotspots(pdb_path, chain, count=2):
    residue_ids = []
    seen = set()
    with open(pdb_path, "r") as handle:
        for line in handle:
            if not line.startswith("ATOM") or len(line) < 27 or line[21].strip() != chain:
                continue
            residue_id = (line[22:26].strip(), line[26].strip())
            if residue_id not in seen:
                seen.add(residue_id)
                residue_ids.append(residue_id)
    if len(residue_ids) < count:
        raise ValueError(f"{pdb_path} has fewer than {count} residues in chain {chain}")
    return ",".join(f"{chain}{number}{insertion}" for number, insertion in residue_ids[:count])

# Active GLP-1(7-36)NH2 conformers: receptor-bound 6X18 plus two solution-NMR conformers from 1D0R.
glp1_6x18 = _download_pdb("6X18")
glp1_1d0r = _download_pdb("1D0R")
active_sources = [
    ("GLP1_7_36_bound", glp1_6x18, "P", 1, "GLP1_7_36_6X18.pdb"),
    ("GLP1_7_36_nmr_1", glp1_1d0r, "A", 1, "GLP1_7_36_1D0R_model1.pdb"),
    ("GLP1_7_36_nmr_10", glp1_1d0r, "A", 10, "GLP1_7_36_1D0R_model10.pdb"),
]
positive_target_panel = []
for name, source, source_chain, model_index, output_name in active_sources:
    target_pdb = _extract_chain(source, source_chain, output_name, model_index=model_index)
    positive_target_panel.append({
        "name": name,
        "pdb": target_pdb,
        "chains": source_chain,
        "hotspots": _first_residue_hotspots(target_pdb, source_chain, count=2),
    })

# DPP-4 product conformers are generated by removing the first two residues from matched active structures.
truncated_bound = _extract_chain(positive_target_panel[0]["pdb"], "P", "GLP1_9_36_6X18.pdb", drop_first_residues=2)
truncated_solution = _extract_chain(positive_target_panel[1]["pdb"], "A", "GLP1_9_36_1D0R_model1.pdb", drop_first_residues=2)

# Homologous negative targets named in the technical document.
gip_source = _download_pdb("7DTY")
glucagon_source = _download_pdb("6LMK")
oxyntomodulin_source = _download_pdb("7LLY")
negative_target_panel = [
    {"name": "GLP1_9_36_bound", "pdb": truncated_bound, "chains": "P"},
    {"name": "GLP1_9_36_solution", "pdb": truncated_solution, "chains": "A"},
    {"name": "GIP", "pdb": _extract_chain(gip_source, "P", "GIP_7DTY.pdb"), "chains": "P"},
    {"name": "Glucagon", "pdb": _extract_chain(glucagon_source, "E", "Glucagon_6LMK.pdb"), "chains": "E"},
    {"name": "Oxyntomodulin", "pdb": _extract_chain(oxyntomodulin_source, "P", "Oxyntomodulin_7LLY.pdb"), "chains": "P"},
]

# The original BindCraft single-target fields remain unchanged in name and meaning; the first active conformer is the default target.
starting_pdb = positive_target_panel[0]["pdb"]
chains = positive_target_panel[0]["chains"]
target_hotspot_residues = positive_target_panel[0]["hotspots"]

# @markdown ---
# @markdown Enter a previous target-settings JSON to continue an existing campaign. New GLP-1 fields are loaded from that file when present.
load_previous_target_settings = "" # @param {"type":"string","placeholder":""}
# @markdown ---

if load_previous_target_settings:
    target_settings_path = load_previous_target_settings
else:
    lengths = [int(x.strip()) for x in lengths.split(',') if len(lengths.split(',')) == 2]
    if len(lengths) != 2:
        raise ValueError("Incorrect specification of binder lengths.")

    settings = {
        "design_path": design_path,
        "binder_name": binder_name,
        "starting_pdb": starting_pdb,
        "chains": chains,
        "target_hotspot_residues": target_hotspot_residues,
        "lengths": lengths,
        "number_of_final_designs": number_of_final_designs,
        "positive_target_panel": positive_target_panel,
        "negative_target_panel": negative_target_panel,
        "minimum_positive_iptm": minimum_positive_iptm,
        "maximum_negative_iptm": maximum_negative_iptm,
        "minimum_selectivity_margin": minimum_selectivity_margin,
        "minimum_n_terminal_contacts": minimum_n_terminal_contacts,
        "n_terminal_contact_cutoff": n_terminal_contact_cutoff,
        "selectivity_validation_models": selectivity_validation_models,
    }

    target_settings_path = os.path.join(design_path, binder_name+".json")
    os.makedirs(design_path, exist_ok=True)
    with open(target_settings_path, 'w') as f:
        json.dump(settings, f, indent=4)

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Binder design settings updated at: {currenttime}")
print(f"New .json file with target settings has been generated in: {target_settings_path}")

In [ ]:
#@title Advanced settings
# @markdown ---
# @markdown Which binder design protocol to run? Default is recommended. "Beta-sheet" promotes the design of more beta sheeted proteins, but requires more sampling. "Peptide" is optimised for helical peptide binders.
design_protocol = "Default" # @param ["Default","Beta-sheet","Peptide"]
# @markdown What prediction protocol to use?. "Default" performs single sequence prediction of the binder. "HardTarget" uses initial guess to improve complex prediction for difficult targets, but might introduce some bias.
prediction_protocol = "Default" # @param ["Default","HardTarget"]
# @markdown What interface design method to use?. "AlphaFold2" is the default, interface is generated by AlphaFold2. "MPNN" uses soluble MPNN to optimise the interface.
interface_protocol = "AlphaFold2" # @param ["AlphaFold2","MPNN"]
# @markdown What target template protocol to use? "Default" allows for limited amount flexibility. "Masked" allows for greater target flexibility on both sidechain and backbone level.
template_protocol = "Default" # @param ["Default","Masked"]
# @markdown ---

if design_protocol == "Default":
    design_protocol_tag = "default_4stage_multimer"
elif design_protocol == "Beta-sheet":
    design_protocol_tag = "betasheet_4stage_multimer"
elif design_protocol == "Peptide":
    design_protocol_tag = "peptide_3stage_multimer"
else:
    raise ValueError(f"Unsupported design protocol")

if interface_protocol == "AlphaFold2":
    interface_protocol_tag = ""
elif interface_protocol == "MPNN":
    interface_protocol_tag = "_mpnn"
else:
    raise ValueError(f"Unsupported interface protocol")

if template_protocol == "Default":
    template_protocol_tag = ""
elif template_protocol == "Masked":
    template_protocol_tag = "_flexible"
else:
    raise ValueError(f"Unsupported template protocol")

if design_protocol in ["Peptide"]:
    prediction_protocol_tag = ""
else:
    if prediction_protocol == "Default":
        prediction_protocol_tag = ""
    elif prediction_protocol == "HardTarget":
        prediction_protocol_tag = "_hardtarget"
    else:
        raise ValueError(f"Unsupported prediction protocol")

advanced_settings_path = "/content/bindcraft/settings_advanced/" + design_protocol_tag + interface_protocol_tag + template_protocol_tag + prediction_protocol_tag + ".json"

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Advanced design settings updated at: {currenttime}")

In [ ]:
#@title Filters
# @markdown ---
# @markdown Which filters for designs to use? "Default" are recommended, "Peptide" are for the design of peptide binders, "Relaxed" are more permissive but may result in fewer experimental successes, "Peptide_Relaxed" are more permissive filters for non-helical peptides, "None" is for benchmarking.
filter_option = "Default" # @param ["Default", "Peptide", "Relaxed", "Peptide_Relaxed", "None"]
# @markdown ---

if filter_option == "Default":
    filter_settings_path = "/content/bindcraft/settings_filters/default_filters.json"
elif filter_option == "Peptide":
    filter_settings_path = "/content/bindcraft/settings_filters/peptide_filters.json"
elif filter_option == "Relaxed":
    filter_settings_path = "/content/bindcraft/settings_filters/relaxed_filters.json"
elif filter_option == "Peptide_Relaxed":
    filter_settings_path = "/content/bindcraft/settings_filters/peptide_relaxed_filters.json"
elif filter_option == "None":
    filter_settings_path = "/content/bindcraft/settings_filters/no_filters.json"
else:
    raise ValueError(f"Unsupported filter type")

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Filter settings updated at: {currenttime}")

# Everything is set, BindCraft is ready to run!

In [ ]:
# @title Import functions and settings
from bindcraft.functions import *
import re

args = {"settings":target_settings_path,
        "filters":filter_settings_path,
        "advanced":advanced_settings_path}

# Check if JAX-capable GPU is available, otherwise exit
check_jax_gpu()

# perform checks of input setting files
settings_path, filters_path, advanced_path = (args["settings"], args["filters"], args["advanced"])

### load settings from JSON
target_settings, advanced_settings, filters = load_json_settings(settings_path, filters_path, advanced_path)

# Backward-compatible defaults keep older settings JSON files loadable while requiring an explicit negative panel for the new objective.
target_settings.setdefault("positive_target_panel", [{
    "name": "primary",
    "pdb": target_settings["starting_pdb"],
    "chains": target_settings["chains"],
    "hotspots": target_settings["target_hotspot_residues"],
}])
target_settings.setdefault("negative_target_panel", [])
target_settings.setdefault("minimum_positive_iptm", 0.50)
target_settings.setdefault("maximum_negative_iptm", 0.35)
target_settings.setdefault("minimum_selectivity_margin", 0.15)
target_settings.setdefault("minimum_n_terminal_contacts", 2)
target_settings.setdefault("n_terminal_contact_cutoff", 5.0)
target_settings.setdefault("selectivity_validation_models", 1)
if not target_settings["negative_target_panel"]:
    raise ValueError("negative_target_panel is required for active GLP-1 conformer-selective design")

settings_file = os.path.basename(settings_path).split('.')[0]
filters_file = os.path.basename(filters_path).split('.')[0]
advanced_file = os.path.basename(advanced_path).split('.')[0]

### load AF2 model settings
design_models, prediction_models, multimer_validation = load_af2_models(advanced_settings["use_multimer_design"])

### perform checks on advanced_settings
bindcraft_folder = "colab"
advanced_settings = perform_advanced_settings_check(advanced_settings, bindcraft_folder)

### generate directories, design path names can be found within the function
design_paths = generate_directories(target_settings["design_path"])

### generate dataframes
trajectory_labels, design_labels, final_labels = generate_dataframe_labels()

trajectory_csv = os.path.join(target_settings["design_path"], 'trajectory_stats.csv')
mpnn_csv = os.path.join(target_settings["design_path"], 'mpnn_design_stats.csv')
final_csv = os.path.join(target_settings["design_path"], 'final_design_stats.csv')
failure_csv = os.path.join(target_settings["design_path"], 'failure_csv.csv')

create_dataframe(trajectory_csv, trajectory_labels)
create_dataframe(mpnn_csv, design_labels)
create_dataframe(final_csv, final_labels)
generate_filter_pass_csv(failure_csv, args["filters"])

# GLP-1-specific screening metrics are kept in a separate CSV so the original BindCraft schemas and filters remain intact.
selectivity_labels = [
    "Design", "Trajectory_Target", "Positive_Min_i_pTM", "Negative_Max_i_pTM",
    "GLP1_9_36_Max_i_pTM", "Selectivity_Margin", "N_Terminal_Contacts",
    "Selection_Score", "Pass", "Panel_Metrics", "Error"
]
selectivity_csv = os.path.join(target_settings["design_path"], "selectivity_stats.csv")
if not os.path.isfile(selectivity_csv):
    pd.DataFrame(columns=selectivity_labels).to_csv(selectivity_csv, index=False)

def _pdb_heavy_atoms_by_residue(pdb_path, chain):
    residues = []
    residue_atoms = {}
    with open(pdb_path, "r") as handle:
        for line in handle:
            if not line.startswith("ATOM") or len(line) < 54 or line[21].strip() != chain:
                continue
            atom_name = line[12:16].strip()
            if atom_name.startswith("H"):
                continue
            residue_key = (line[22:26].strip(), line[26].strip())
            if residue_key not in residue_atoms:
                residues.append(residue_key)
                residue_atoms[residue_key] = []
            residue_atoms[residue_key].append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return residues, residue_atoms

def count_n_terminal_contacts(complex_pdb, target_chain="A", binder_chain="B", residue_count=2, cutoff=5.0):
    target_residues, target_atoms = _pdb_heavy_atoms_by_residue(complex_pdb, target_chain)
    _, binder_atoms_by_residue = _pdb_heavy_atoms_by_residue(complex_pdb, binder_chain)
    if len(target_residues) < residue_count or not binder_atoms_by_residue:
        return 0
    binder_atoms = np.asarray([atom for atoms in binder_atoms_by_residue.values() for atom in atoms], dtype=float)
    contacts = 0
    cutoff_squared = float(cutoff) ** 2
    for residue_key in target_residues[:residue_count]:
        atoms = np.asarray(target_atoms[residue_key], dtype=float)
        squared_distances = np.sum((atoms[:, None, :] - binder_atoms[None, :, :]) ** 2, axis=-1)
        contacts += int(np.any(squared_distances <= cutoff_squared))
    return contacts

def predict_selectivity_targets(prediction_model, binder_sequence, length, target_panel, model_numbers):
    panel_metrics = {}
    clean_sequence = re.sub("[^A-Z]", "", binder_sequence.upper())
    for target in target_panel:
        prediction_model.prep_inputs(
            pdb_filename=target["pdb"], chain=target["chains"], binder_len=length,
            rm_target_seq=advanced_settings["rm_template_seq_predict"],
            rm_target_sc=advanced_settings["rm_template_sc_predict"]
        )
        model_metrics = []
        for model_num in model_numbers:
            prediction_model.predict(
                seq=clean_sequence, models=[model_num],
                num_recycles=advanced_settings["num_recycles_validation"], verbose=False
            )
            log = copy_dict(prediction_model.aux["log"])
            model_metrics.append({
                "i_pTM": float(log["i_ptm"]),
                "i_pAE": float(log["i_pae"]),
                "pLDDT": float(log["plddt"]),
            })
        panel_metrics[target["name"]] = {
            metric: round(float(np.mean([entry[metric] for entry in model_metrics])), 4)
            for metric in ("i_pTM", "i_pAE", "pLDDT")
        }
    return panel_metrics

def evaluate_glp1_selectivity(prediction_model, binder_sequence, length, primary_target, primary_iptm, best_model_pdb):
    try:
        positive_targets = [
            target for target in target_settings["positive_target_panel"]
            if target["name"] != primary_target["name"]
        ]
        panel_targets = positive_targets + target_settings["negative_target_panel"]
        n_models = max(1, int(target_settings["selectivity_validation_models"]))
        panel_model_numbers = list(prediction_models)[:n_models]
        panel_metrics = predict_selectivity_targets(
            prediction_model, binder_sequence, length, panel_targets, panel_model_numbers
        )

        positive_scores = [float(primary_iptm)] + [panel_metrics[target["name"]]["i_pTM"] for target in positive_targets]
        negative_scores = [panel_metrics[target["name"]]["i_pTM"] for target in target_settings["negative_target_panel"]]
        truncated_scores = [
            panel_metrics[target["name"]]["i_pTM"] for target in target_settings["negative_target_panel"]
            if target["name"].startswith("GLP1_9_36")
        ]
        positive_min = min(positive_scores)
        negative_max = max(negative_scores)
        truncated_max = max(truncated_scores)
        selectivity_margin = positive_min - negative_max
        n_terminal_contacts = count_n_terminal_contacts(
            best_model_pdb, residue_count=2,
            cutoff=target_settings["n_terminal_contact_cutoff"]
        )
        passed = (
            positive_min >= target_settings["minimum_positive_iptm"]
            and negative_max <= target_settings["maximum_negative_iptm"]
            and selectivity_margin >= target_settings["minimum_selectivity_margin"]
            and n_terminal_contacts >= target_settings["minimum_n_terminal_contacts"]
        )
        selection_score = selectivity_margin + 0.025 * n_terminal_contacts
        return {
            "Positive_Min_i_pTM": round(positive_min, 4),
            "Negative_Max_i_pTM": round(negative_max, 4),
            "GLP1_9_36_Max_i_pTM": round(truncated_max, 4),
            "Selectivity_Margin": round(selectivity_margin, 4),
            "N_Terminal_Contacts": int(n_terminal_contacts),
            "Selection_Score": round(selection_score, 4),
            "Pass": bool(passed),
            "Panel_Metrics": json.dumps(panel_metrics, sort_keys=True),
            "Error": "",
        }
    except Exception as exc:
        return {
            "Positive_Min_i_pTM": np.nan, "Negative_Max_i_pTM": np.nan,
            "GLP1_9_36_Max_i_pTM": np.nan, "Selectivity_Margin": np.nan,
            "N_Terminal_Contacts": 0, "Selection_Score": -np.inf,
            "Pass": False, "Panel_Metrics": "", "Error": str(exc),
        }

def record_selectivity_result(design_name, trajectory_target, metrics):
    row = {"Design": design_name, "Trajectory_Target": trajectory_target, **metrics}
    pd.DataFrame([row], columns=selectivity_labels).to_csv(
        selectivity_csv, mode="a", header=False, index=False
    )

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Loaded design functions and settings at: {currenttime}")

In [ ]:
#@title Initialise PyRosetta

####################################
####################################
####################################
### initialise PyRosetta
pr.init(f'-ignore_unrecognized_res -ignore_zero_occupancy -mute all -holes:dalphaball {advanced_settings["dalphaball_path"]} -corrections::beta_nov16 true -relax:default_repeats 1')

In [ ]:
#@title Run BindCraft!
####################################
###################### BindCraft Run
####################################
# Colab-specific: live displays
num_sampled_trajectories = len(pd.read_csv(trajectory_csv))
num_accepted_designs = len(pd.read_csv(final_csv))
sampled_trajectories_label = HTML(value=f"<h3 style='color: #1f77b4;'>Sampled Trajectories: <span style='color: #1f77b4;'>{num_sampled_trajectories}</span></h3>")
accepted_designs_label = HTML(value=f"<h3 style='color: #2ca02c;'>Accepted Designs: <span style='color: #2ca02c;'>{num_accepted_designs}</span></h3>")
display(VBox([sampled_trajectories_label, accepted_designs_label]))

# initialise counters
script_start_time = time.time()
trajectory_n = 1
accepted_designs = 0

### start design loop
while True:
    ### check if we have the target number of binders
    final_designs_reached = check_accepted_designs(design_paths, mpnn_csv, final_labels, final_csv, advanced_settings, target_settings, design_labels)

    if final_designs_reached:
        # stop design loop execution
        break

    ### check if we reached maximum allowed trajectories
    max_trajectories_reached = check_n_trajectories(design_paths, advanced_settings)

    if max_trajectories_reached:
        break

    ### Initialise design
    # measure time to generate design
    trajectory_start_time = time.time()

    # generate random seed to vary designs
    seed = int(np.random.randint(0, high=999999, size=1, dtype=int)[0])

    # Rotate the hallucination target through the active conformer ensemble so conformational robustness enters generation, not only ranking.
    trajectory_target = target_settings["positive_target_panel"][(trajectory_n - 1) % len(target_settings["positive_target_panel"])]
    trajectory_target_pdb = trajectory_target["pdb"]
    trajectory_target_chains = trajectory_target["chains"]
    trajectory_target_hotspots = trajectory_target["hotspots"]

    # sample binder design length randomly from defined distribution
    samples = np.arange(min(target_settings["lengths"]), max(target_settings["lengths"]) + 1)
    length = np.random.choice(samples)

    # load desired helicity value to sample different secondary structure contents
    helicity_value = load_helicity(advanced_settings)

    # generate design name and check if same trajectory was already run
    design_name = target_settings["binder_name"] + "_l" + str(length) + "_s"+ str(seed)
    trajectory_dirs = ["Trajectory", "Trajectory/Relaxed", "Trajectory/LowConfidence", "Trajectory/Clashing"]
    trajectory_exists = any(os.path.exists(os.path.join(design_paths[trajectory_dir], design_name + ".pdb")) for trajectory_dir in trajectory_dirs)

    if not trajectory_exists:
        print("Starting trajectory: "+design_name)

        ### Begin binder hallucination
        trajectory = binder_hallucination(design_name, trajectory_target_pdb, trajectory_target_chains,
                                            trajectory_target_hotspots, length, seed, helicity_value,
                                            design_models, advanced_settings, design_paths, failure_csv)
        trajectory_metrics = copy_dict(trajectory._tmp["best"]["aux"]["log"]) # contains plddt, ptm, i_ptm, pae, i_pae
        trajectory_pdb = os.path.join(design_paths["Trajectory"], design_name + ".pdb")

        # round the metrics to two decimal places
        trajectory_metrics = {k: round(v, 2) if isinstance(v, float) else v for k, v in trajectory_metrics.items()}

        # time trajectory
        trajectory_time = time.time() - trajectory_start_time
        trajectory_time_text = f"{'%d hours, %d minutes, %d seconds' % (int(trajectory_time // 3600), int((trajectory_time % 3600) // 60), int(trajectory_time % 60))}"
        print("Starting trajectory took: "+trajectory_time_text)
        print("")

        # Proceed if there is no trajectory termination signal
        if trajectory.aux["log"]["terminate"] == "":
            # Relax binder to calculate statistics
            trajectory_relaxed = os.path.join(design_paths["Trajectory/Relaxed"], design_name + ".pdb")
            pr_relax(trajectory_pdb, trajectory_relaxed)

            # define binder chain, placeholder in case multi-chain parsing in ColabDesign gets changed
            binder_chain = "B"

            # Calculate clashes before and after relaxation
            num_clashes_trajectory = calculate_clash_score(trajectory_pdb)
            num_clashes_relaxed = calculate_clash_score(trajectory_relaxed)

            # secondary structure content of starting trajectory binder and interface
            trajectory_alpha, trajectory_beta, trajectory_loops, trajectory_alpha_interface, trajectory_beta_interface, trajectory_loops_interface, trajectory_i_plddt, trajectory_ss_plddt = calc_ss_percentage(trajectory_pdb, advanced_settings, binder_chain)

            # analyze interface scores for relaxed af2 trajectory
            trajectory_interface_scores, trajectory_interface_AA, trajectory_interface_residues = score_interface(trajectory_relaxed, binder_chain)

            # starting binder sequence
            trajectory_sequence = trajectory.get_seq(get_best=True)[0]

            # analyze sequence
            traj_seq_notes = validate_design_sequence(trajectory_sequence, num_clashes_relaxed, advanced_settings)

            # target structure RMSD compared to input PDB
            trajectory_target_rmsd = unaligned_rmsd(trajectory_target_pdb, trajectory_pdb, trajectory_target_chains, 'A')

            # save trajectory statistics into CSV
            trajectory_data = [design_name, advanced_settings["design_algorithm"], length, seed, helicity_value, trajectory_target_hotspots, trajectory_sequence, trajectory_interface_residues,
                                trajectory_metrics['plddt'], trajectory_metrics['ptm'], trajectory_metrics['i_ptm'], trajectory_metrics['pae'], trajectory_metrics['i_pae'],
                                trajectory_i_plddt, trajectory_ss_plddt, num_clashes_trajectory, num_clashes_relaxed, trajectory_interface_scores['binder_score'],
                                trajectory_interface_scores['surface_hydrophobicity'], trajectory_interface_scores['interface_sc'], trajectory_interface_scores['interface_packstat'],
                                trajectory_interface_scores['interface_dG'], trajectory_interface_scores['interface_dSASA'], trajectory_interface_scores['interface_dG_SASA_ratio'],
                                trajectory_interface_scores['interface_fraction'], trajectory_interface_scores['interface_hydrophobicity'], trajectory_interface_scores['interface_nres'], trajectory_interface_scores['interface_interface_hbonds'],
                                trajectory_interface_scores['interface_hbond_percentage'], trajectory_interface_scores['interface_delta_unsat_hbonds'], trajectory_interface_scores['interface_delta_unsat_hbonds_percentage'],
                                trajectory_alpha_interface, trajectory_beta_interface, trajectory_loops_interface, trajectory_alpha, trajectory_beta, trajectory_loops, trajectory_interface_AA, trajectory_target_rmsd,
                                trajectory_time_text, traj_seq_notes, settings_file, filters_file, advanced_file]
            insert_data(trajectory_csv, trajectory_data)

            if advanced_settings["enable_mpnn"]:
                # initialise MPNN counters
                mpnn_n = 1
                accepted_mpnn = 0
                mpnn_dict = {}
                design_start_time = time.time()

                ### MPNN redesign of starting binder
                mpnn_trajectories = mpnn_gen_sequence(trajectory_pdb, binder_chain, trajectory_interface_residues, advanced_settings)
                existing_mpnn_sequences = set(pd.read_csv(mpnn_csv, usecols=['Sequence'])['Sequence'].values)

                # create set of MPNN sequences with allowed amino acid composition
                restricted_AAs = set(aa.strip().upper() for aa in advanced_settings["omit_AAs"].split(',')) if advanced_settings["force_reject_AA"] else set()

                mpnn_sequences = sorted({
                    mpnn_trajectories['seq'][n][-length:]: {
                        'seq': mpnn_trajectories['seq'][n][-length:],
                        'score': mpnn_trajectories['score'][n],
                        'seqid': mpnn_trajectories['seqid'][n]
                    } for n in range(advanced_settings["num_seqs"])
                    if (not restricted_AAs or not any(aa in mpnn_trajectories['seq'][n][-length:].upper() for aa in restricted_AAs))
                    and mpnn_trajectories['seq'][n][-length:] not in existing_mpnn_sequences
                }.values(), key=lambda x: x['score'])

                del existing_mpnn_sequences

                # check whether any sequences are left after amino acid rejection and duplication check, and if yes proceed with prediction
                if mpnn_sequences:
                    # add optimisation for increasing recycles if trajectory is beta sheeted
                    if advanced_settings["optimise_beta"] and float(trajectory_beta) > 15:
                        advanced_settings["num_recycles_validation"] = advanced_settings["optimise_beta_recycles_valid"]

                    ### Compile prediction models once for faster prediction of MPNN sequences
                    clear_mem()
                    # compile complex prediction model
                    complex_prediction_model = mk_afdesign_model(protocol="binder", num_recycles=advanced_settings["num_recycles_validation"], data_dir=advanced_settings["af_params_dir"],
                                                                use_multimer=multimer_validation)
                    complex_prediction_model.prep_inputs(pdb_filename=trajectory_target_pdb, chain=trajectory_target_chains, binder_len=length, rm_target_seq=advanced_settings["rm_template_seq_predict"],
                                                        rm_target_sc=advanced_settings["rm_template_sc_predict"])

                    # compile binder monomer prediction model
                    binder_prediction_model = mk_afdesign_model(protocol="hallucination", use_templates=False, initial_guess=False,
                                                                use_initial_atom_pos=False, num_recycles=advanced_settings["num_recycles_validation"],
                                                                data_dir=advanced_settings["af_params_dir"], use_multimer=multimer_validation)
                    binder_prediction_model.prep_inputs(length=length)

                    # iterate over designed sequences
                    for mpnn_sequence in mpnn_sequences:
                        mpnn_time = time.time()

                        # generate mpnn design name numbering
                        mpnn_design_name = design_name + "_mpnn" + str(mpnn_n)
                        mpnn_score = round(mpnn_sequence['score'],2)
                        mpnn_seqid = round(mpnn_sequence['seqid'],2)

                        # add design to dictionary
                        mpnn_dict[mpnn_design_name] = {'seq': mpnn_sequence['seq'], 'score': mpnn_score, 'seqid': mpnn_seqid}

                        # save fasta sequence
                        if advanced_settings["save_mpnn_fasta"] is True:
                            save_fasta(mpnn_design_name, mpnn_sequence['seq'], design_paths)

                        ### Predict mpnn redesigned binder complex using masked templates
                        # The same model is reused for counter-screening; restore the current positive conformer before each base prediction.
                        complex_prediction_model.prep_inputs(
                            pdb_filename=trajectory_target_pdb, chain=trajectory_target_chains, binder_len=length,
                            rm_target_seq=advanced_settings["rm_template_seq_predict"],
                            rm_target_sc=advanced_settings["rm_template_sc_predict"]
                        )
                        mpnn_complex_statistics, pass_af2_filters = predict_binder_complex(complex_prediction_model,
                                                                                        mpnn_sequence['seq'], mpnn_design_name,
                                                                                        trajectory_target_pdb, trajectory_target_chains,
                                                                                        length, trajectory_pdb, prediction_models, advanced_settings,
                                                                                        filters, design_paths, failure_csv)

                        # if AF2 filters are not passed then skip the scoring
                        if not pass_af2_filters:
                            print(f"Base AF2 filters not passed for {mpnn_design_name}, skipping interface scoring")
                            mpnn_n += 1
                            continue

                        # calculate statistics for each model individually
                        for model_num in prediction_models:
                            mpnn_design_pdb = os.path.join(design_paths["MPNN"], f"{mpnn_design_name}_model{model_num+1}.pdb")
                            mpnn_design_relaxed = os.path.join(design_paths["MPNN/Relaxed"], f"{mpnn_design_name}_model{model_num+1}.pdb")

                            if os.path.exists(mpnn_design_pdb):
                                # Calculate clashes before and after relaxation
                                num_clashes_mpnn = calculate_clash_score(mpnn_design_pdb)
                                num_clashes_mpnn_relaxed = calculate_clash_score(mpnn_design_relaxed)

                                # analyze interface scores for relaxed af2 trajectory
                                mpnn_interface_scores, mpnn_interface_AA, mpnn_interface_residues = score_interface(mpnn_design_relaxed, binder_chain)

                                # secondary structure content of starting trajectory binder
                                mpnn_alpha, mpnn_beta, mpnn_loops, mpnn_alpha_interface, mpnn_beta_interface, mpnn_loops_interface, mpnn_i_plddt, mpnn_ss_plddt = calc_ss_percentage(mpnn_design_pdb, advanced_settings, binder_chain)

                                # unaligned RMSD calculate to determine if binder is in the designed binding site
                                rmsd_site = unaligned_rmsd(trajectory_pdb, mpnn_design_pdb, binder_chain, binder_chain)

                                # calculate RMSD of target compared to input PDB
                                target_rmsd = target_pdb_rmsd(mpnn_design_pdb, trajectory_target_pdb, trajectory_target_chains)

                                # add the additional statistics to the mpnn_complex_statistics dictionary
                                mpnn_complex_statistics[model_num+1].update({
                                    'i_pLDDT': mpnn_i_plddt,
                                    'ss_pLDDT': mpnn_ss_plddt,
                                    'Unrelaxed_Clashes': num_clashes_mpnn,
                                    'Relaxed_Clashes': num_clashes_mpnn_relaxed,
                                    'Binder_Energy_Score': mpnn_interface_scores['binder_score'],
                                    'Surface_Hydrophobicity': mpnn_interface_scores['surface_hydrophobicity'],
                                    'ShapeComplementarity': mpnn_interface_scores['interface_sc'],
                                    'PackStat': mpnn_interface_scores['interface_packstat'],
                                    'dG': mpnn_interface_scores['interface_dG'],
                                    'dSASA': mpnn_interface_scores['interface_dSASA'],
                                    'dG/dSASA': mpnn_interface_scores['interface_dG_SASA_ratio'],
                                    'Interface_SASA_%': mpnn_interface_scores['interface_fraction'],
                                    'Interface_Hydrophobicity': mpnn_interface_scores['interface_hydrophobicity'],
                                    'n_InterfaceResidues': mpnn_interface_scores['interface_nres'],
                                    'n_InterfaceHbonds': mpnn_interface_scores['interface_interface_hbonds'],
                                    'InterfaceHbondsPercentage': mpnn_interface_scores['interface_hbond_percentage'],
                                    'n_InterfaceUnsatHbonds': mpnn_interface_scores['interface_delta_unsat_hbonds'],
                                    'InterfaceUnsatHbondsPercentage': mpnn_interface_scores['interface_delta_unsat_hbonds_percentage'],
                                    'InterfaceAAs': mpnn_interface_AA,
                                    'Interface_Helix%': mpnn_alpha_interface,
                                    'Interface_BetaSheet%': mpnn_beta_interface,
                                    'Interface_Loop%': mpnn_loops_interface,
                                    'Binder_Helix%': mpnn_alpha,
                                    'Binder_BetaSheet%': mpnn_beta,
                                    'Binder_Loop%': mpnn_loops,
                                    'Hotspot_RMSD': rmsd_site,
                                    'Target_RMSD': target_rmsd
                                })

                                # save space by removing unrelaxed predicted mpnn complex pdb?
                                if advanced_settings["remove_unrelaxed_complex"]:
                                    os.remove(mpnn_design_pdb)

                        # calculate complex averages
                        mpnn_complex_averages = calculate_averages(mpnn_complex_statistics, handle_aa=True)

                        ### Predict binder alone in single sequence mode
                        binder_statistics = predict_binder_alone(binder_prediction_model, mpnn_sequence['seq'], mpnn_design_name, length,
                                                                trajectory_pdb, binder_chain, prediction_models, advanced_settings, design_paths)

                        # extract RMSDs of binder to the original trajectory
                        for model_num in prediction_models:
                            mpnn_binder_pdb = os.path.join(design_paths["MPNN/Binder"], f"{mpnn_design_name}_model{model_num+1}.pdb")

                            if os.path.exists(mpnn_binder_pdb):
                                rmsd_binder = unaligned_rmsd(trajectory_pdb, mpnn_binder_pdb, binder_chain, "A")

                            # append to statistics
                            binder_statistics[model_num+1].update({
                                    'Binder_RMSD': rmsd_binder
                                })

                            # save space by removing binder monomer models?
                            if advanced_settings["remove_binder_monomer"]:
                                os.remove(mpnn_binder_pdb)

                        # calculate binder averages
                        binder_averages = calculate_averages(binder_statistics)

                        # analyze sequence to make sure there are no cysteins and it contains residues that absorb UV for detection
                        seq_notes = validate_design_sequence(mpnn_sequence['seq'], mpnn_complex_averages.get('Relaxed_Clashes', None), advanced_settings)

                        # measure time to generate design
                        mpnn_end_time = time.time() - mpnn_time
                        elapsed_mpnn_text = f"{'%d hours, %d minutes, %d seconds' % (int(mpnn_end_time // 3600), int((mpnn_end_time % 3600) // 60), int(mpnn_end_time % 60))}"


                        # Insert statistics about MPNN design into CSV, will return None if corresponding model does note exist
                        model_numbers = range(1, 6)
                        statistics_labels = ['pLDDT', 'pTM', 'i_pTM', 'pAE', 'i_pAE', 'i_pLDDT', 'ss_pLDDT', 'Unrelaxed_Clashes', 'Relaxed_Clashes', 'Binder_Energy_Score', 'Surface_Hydrophobicity',
                                            'ShapeComplementarity', 'PackStat', 'dG', 'dSASA', 'dG/dSASA', 'Interface_SASA_%', 'Interface_Hydrophobicity', 'n_InterfaceResidues', 'n_InterfaceHbonds', 'InterfaceHbondsPercentage',
                                            'n_InterfaceUnsatHbonds', 'InterfaceUnsatHbondsPercentage', 'Interface_Helix%', 'Interface_BetaSheet%', 'Interface_Loop%', 'Binder_Helix%',
                                            'Binder_BetaSheet%', 'Binder_Loop%', 'InterfaceAAs', 'Hotspot_RMSD', 'Target_RMSD']

                        # Initialize mpnn_data with the non-statistical data
                        mpnn_data = [mpnn_design_name, advanced_settings["design_algorithm"], length, seed, helicity_value, trajectory_target_hotspots, mpnn_sequence['seq'], mpnn_interface_residues, mpnn_score, mpnn_seqid]

                        # Add the statistical data for mpnn_complex
                        for label in statistics_labels:
                            mpnn_data.append(mpnn_complex_averages.get(label, None))
                            for model in model_numbers:
                                mpnn_data.append(mpnn_complex_statistics.get(model, {}).get(label, None))

                        # Add the statistical data for binder
                        for label in ['pLDDT', 'pTM', 'pAE', 'Binder_RMSD']:  # These are the labels for binder alone
                            mpnn_data.append(binder_averages.get(label, None))
                            for model in model_numbers:
                                mpnn_data.append(binder_statistics.get(model, {}).get(label, None))

                        # Add the remaining non-statistical data
                        mpnn_data.extend([elapsed_mpnn_text, seq_notes, settings_file, filters_file, advanced_file])

                        # insert data into csv
                        insert_data(mpnn_csv, mpnn_data)

                        # find best model number by pLDDT
                        plddt_values = {i: mpnn_data[i] for i in range(11, 15) if mpnn_data[i] is not None}

                        # Find the key with the highest value
                        highest_plddt_key = int(max(plddt_values, key=plddt_values.get))

                        # Output the number part of the key
                        best_model_number = highest_plddt_key - 10
                        best_model_pdb = os.path.join(design_paths["MPNN/Relaxed"], f"{mpnn_design_name}_model{best_model_number}.pdb")

                        # run design data against filter thresholds
                        filter_conditions = check_filters(mpnn_data, design_labels, filters)
                        if filter_conditions == True:
                            selectivity_metrics = evaluate_glp1_selectivity(
                                complex_prediction_model, mpnn_sequence['seq'], length,
                                trajectory_target, mpnn_complex_averages['i_pTM'], best_model_pdb
                            )
                            record_selectivity_result(mpnn_design_name, trajectory_target["name"], selectivity_metrics)

                            if selectivity_metrics["Pass"]:
                                print(mpnn_design_name+" passed original and GLP-1 selectivity filters")
                                accepted_mpnn += 1
                                accepted_designs += 1

                                # copy designs to accepted folder
                                shutil.copy(best_model_pdb, design_paths["Accepted"])

                                # insert data into final csv
                                final_data = [''] + mpnn_data
                                insert_data(final_csv, final_data)

                                # copy animation from accepted trajectory
                                if advanced_settings["save_design_animations"]:
                                    accepted_animation = os.path.join(design_paths["Accepted/Animation"], f"{design_name}.html")
                                    if not os.path.exists(accepted_animation):
                                        shutil.copy(os.path.join(design_paths["Trajectory/Animation"], f"{design_name}.html"), accepted_animation)

                                # copy plots of accepted trajectory
                                plot_files = os.listdir(design_paths["Trajectory/Plots"])
                                plots_to_copy = [f for f in plot_files if f.startswith(design_name) and f.endswith('.png')]
                                for accepted_plot in plots_to_copy:
                                    source_plot = os.path.join(design_paths["Trajectory/Plots"], accepted_plot)
                                    target_plot = os.path.join(design_paths["Accepted/Plots"], accepted_plot)
                                    if not os.path.exists(target_plot):
                                        shutil.copy(source_plot, target_plot)
                            else:
                                print(f"{mpnn_design_name} failed GLP-1 selectivity gate: {selectivity_metrics}")
                                shutil.copy(best_model_pdb, design_paths["Rejected"])

                        else:
                            print(f"Unmet filter conditions for {mpnn_design_name}")
                            failure_df = pd.read_csv(failure_csv)
                            special_prefixes = ('Average_', '1_', '2_', '3_', '4_', '5_')
                            incremented_columns = set()

                            for column in filter_conditions:
                                base_column = column
                                for prefix in special_prefixes:
                                    if column.startswith(prefix):
                                        base_column = column.split('_', 1)[1]

                                if base_column not in incremented_columns:
                                    failure_df[base_column] = failure_df[base_column] + 1
                                    incremented_columns.add(base_column)

                            failure_df.to_csv(failure_csv, index=False)
                            shutil.copy(best_model_pdb, design_paths["Rejected"])

                        # increase MPNN design number
                        mpnn_n += 1

                        # if enough mpnn sequences of the same trajectory pass filters then stop
                        if accepted_mpnn >= advanced_settings["max_mpnn_sequences"]:
                            break

                    if accepted_mpnn >= 1:
                        print("Found "+str(accepted_mpnn)+" MPNN designs passing filters")
                    else:
                        print("No accepted MPNN designs found for this trajectory.")

                else:
                    print('Duplicate MPNN designs sampled with different trajectory, skipping current trajectory optimisation')

                # save space by removing unrelaxed design trajectory PDB
                if advanced_settings["remove_unrelaxed_trajectory"]:
                    os.remove(trajectory_pdb)

                # measure time it took to generate designs for one trajectory
                design_time = time.time() - design_start_time
                design_time_text = f"{'%d hours, %d minutes, %d seconds' % (int(design_time // 3600), int((design_time % 3600) // 60), int(design_time % 60))}"
                print("Design and validation of trajectory "+design_name+" took: "+design_time_text)

            # analyse the rejection rate of trajectories to see if we need to readjust the design weights
            if trajectory_n >= advanced_settings["start_monitoring"] and advanced_settings["enable_rejection_check"]:
                acceptance = accepted_designs / trajectory_n
                if not acceptance >= advanced_settings["acceptance_rate"]:
                    print("The ratio of successful designs is lower than defined acceptance rate! Consider changing your design settings!")
                    print("Script execution stopping...")
                    break

        # increase trajectory number
        trajectory_n += 1

        # Colab-specific: update counters
        num_sampled_trajectories = len(pd.read_csv(trajectory_csv))
        num_accepted_designs = len(pd.read_csv(final_csv))
        sampled_trajectories_label.value = f"Sampled trajectories: {num_sampled_trajectories}"
        accepted_designs_label.value = f"Accepted designs: {num_accepted_designs}"

### Script finished
elapsed_time = time.time() - script_start_time
elapsed_text = f"{'%d hours, %d minutes, %d seconds' % (int(elapsed_time // 3600), int((elapsed_time % 3600) // 60), int(elapsed_time % 60))}"
print("Finished all designs. Script execution for "+str(trajectory_n)+" trajectories took: "+elapsed_text)

In [ ]:
#@title Consolidate & Rank Designs
#@markdown ---
accepted_binders = [f for f in os.listdir(design_paths["Accepted"]) if f.endswith('.pdb')]

for f in os.listdir(design_paths["Accepted/Ranked"]):
    os.remove(os.path.join(design_paths["Accepted/Ranked"], f))

# Load the original BindCraft metrics and the GLP-1-specific selectivity metrics.
design_df = pd.read_csv(mpnn_csv)
selectivity_df = pd.read_csv(selectivity_csv)
selectivity_df = selectivity_df[selectivity_df["Pass"].astype(str).str.lower().eq("true")]
selectivity_df = selectivity_df.sort_values("Selection_Score").drop_duplicates("Design", keep="last")
design_df = design_df.merge(selectivity_df, on="Design", how="inner")
design_df = design_df.sort_values(
    ["Selection_Score", "Positive_Min_i_pTM", "Average_i_pTM"],
    ascending=[False, False, False]
)

# Preserve every original final column and append the new selection evidence.
selection_output_labels = [
    label for label in selectivity_labels
    if label not in {"Design", "Pass", "Panel_Metrics", "Error"}
]
ranked_labels = final_labels + [label for label in selection_output_labels if label not in final_labels]
final_df = pd.DataFrame(columns=ranked_labels)

# Check the ranking of the designs and copy them with new ranked IDs to the folder.
rank = 1
for _, row in design_df.iterrows():
    for binder in accepted_binders:
        design_id, model = binder.rsplit('_model', 1)
        if design_id == row['Design']:
            row_data = {
                'Rank': rank,
                **{label: row[label] for label in design_labels},
                **{label: row[label] for label in selection_output_labels},
            }
            final_df = pd.concat([final_df, pd.DataFrame([row_data])], ignore_index=True)
            old_path = os.path.join(design_paths["Accepted"], binder)
            new_path = os.path.join(design_paths["Accepted/Ranked"], f"{rank}_{design_id}_model{model.rsplit('.', 1)[0]}.pdb")
            shutil.copyfile(old_path, new_path)
            rank += 1
            break

final_df.to_csv(final_csv, index=False)
print("Designs ranked by active-GLP-1 selectivity and final_design_stats.csv generated")

In [ ]:
#@title Top 20 Designs
df = pd.read_csv(os.path.join(design_path, 'final_design_stats.csv'))
df.head(20)

In [ ]:
#@title Top Design Display
import py3Dmol
import glob
from IPython.display import HTML

#### pymol top design
top_design_dir = os.path.join(design_path, 'Accepted', 'Ranked')
top_design_pdb = glob.glob(os.path.join(top_design_dir, '1_*.pdb'))[0]

# Visualise in PyMOL
view = py3Dmol.view()
view.addModel(open(top_design_pdb, 'r').read(),'pdb')
view.setBackgroundColor('white')
view.setStyle({'chain':'A'}, {'cartoon': {'color':'#3c5b6f'}})
view.setStyle({'chain':'B'}, {'cartoon': {'color':'#B76E79'}})
view.zoomTo()
view.show()

In [ ]:
#@title Display animation
import glob
from IPython.display import HTML

#### pymol top design
top_design_dir = os.path.join(design_path, 'Accepted', 'Ranked')
top_design_pdb = glob.glob(os.path.join(top_design_dir, '1_*.pdb'))[0]

top_design_name = os.path.basename(top_design_pdb).split('1_', 1)[1].split('_mpnn')[0]
top_design_animation = os.path.join(design_path, 'Accepted', 'Animation', f"{top_design_name}.html")

# Show animation
HTML(top_design_animation)